# Milestone 3 — Silver Market Prices

Goal:
- Read Bronze table
- Clean and standardize data
- Use df.transform()
- Create silver_market_prices table

In [0]:
import yaml
import sys

from transforms.market_price_transforms import clean_market_prices
sys.path.append("/Workspace/Repos/Mini Projects/Vattenfall-Engineering-Capstone-Project/src")

In [0]:
from transforms.market_price_transforms import clean_market_prices


In [0]:
import yaml

config_path = "/Workspace/Repos/Mini Projects/Vattenfall-Engineering-Capstone-Project/config/project_config.yml"

with open(config_path, "r") as f:
    config = yaml.safe_load(f)

catalog = config["catalog"]
raw_schema = config["schemas"]["raw"]
refined_schema = config["schemas"]["refined"]

print(catalog, raw_schema, refined_schema)

In [0]:
df_bronze = spark.table(f"{catalog}.{raw_schema}.bronze_market_prices")
display(df_bronze)

In [0]:
df_silver = df_bronze.transform(clean_market_prices)

In [0]:
display(df_bronze)

In [0]:
df_silver = (
    df_silver
    .withColumn("report_day", F.to_date("event_date"))
    .withColumn("market_type", F.upper(F.col("market_type")))
    .withColumn("region", F.upper(F.col("region")))
    .withColumn("price_eur_mwh", F.col("price_eur_mwh").cast("double"))
    .withColumn("volume_mwh", F.col("volume_mwh").cast("double"))
    .filter(F.col("price_eur_mwh").isNotNull())
)

## Final Silver Column Selection

In this step we keep only the cleaned and business-relevant columns for the Silver layer.

We remove:
- `_rescued_data` → debugging/Bronze-only column
- duplicate or unnecessary fields

This makes the Silver table:
- cleaner
- standardized
- easier for downstream analytics and Gold layer usage

In [0]:
cols_to_drop = [c for c in df_silver.columns if "rescued" in c.lower()]

df_silver = df_silver.drop(*cols_to_drop)

In [0]:
display(df_silver)

df_silver.printSchema()

print("Row Count:", df_silver.count())

In [0]:
df_silver.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{refined_schema}.silver_market_prices")

In [0]:
display(
    spark.table(f"{catalog}.{refined_schema}.silver_market_prices")
)

#RESULT

- clean Silver dataset
- no invalid rows
- no schema issues
- no hardcoding risky columns
- proper Bronze → Silver transformation
